# Import and Clean Low Carbon London Data

**Author**: Sylvia Hipp

Time Series Analysis Final Project

Data Source: https://data.london.gov.uk/dataset/smartmeter-energy-consumption-data-in-london-households-vqm0d/


## Setup Environment

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

os.chdir(r"C:\Users\sch90\Documents\Repositories\GualtieriReedHippKastenSalzberg_ENV797_TSA_FinalProject")

## Download and Wrangle Dataset

Import Data

In [2]:
# get path for all LCL files
lcl_path = Path(os.path.join("data", "raw", "Partitioned LCL Data", "Small LCL Data"))
lcl_files = os.listdir(lcl_path)
lcl_files = [os.path.join(lcl_path, file) for file in lcl_files]

# read CSVs and combine into one df
lcl_dfs = list(map(pd.read_csv, lcl_files))
lcl_df = pd.concat(lcl_dfs).reset_index(drop=True)
lcl_df.columns = ["lcl_id", "std_or_tou", "datetime", "kwh_per_hh"]

In [7]:
lcl_df

,lcl_id,std_or_tou,datetime,kwh_per_hh,datetime_utc
0,MAC000002,Std,2012-10-12 00:30:00,0,2012-10-12 00:30:00+00:00
1,MAC000002,Std,2012-10-12 01:00:00,0,2012-10-12 01:00:00+00:00
2,MAC000002,Std,2012-10-12 01:30:00,0,2012-10-12 01:30:00+00:00
3,MAC000002,Std,2012-10-12 02:00:00,0,2012-10-12 02:00:00+00:00
4,MAC000002,Std,2012-10-12 02:30:00,0,2012-10-12 02:30:00+00:00
...,...,...,...,...,...
167932469,MAC004268,Std,2013-06-29 07:00:00,0.151,2013-06-29 07:00:00+00:00
167932470,MAC004268,Std,2013-06-29 07:30:00,0.278,2013-06-29 07:30:00+00:00
167932471,MAC004268,Std,2013-06-29 08:00:00,0.128,2013-06-29 08:00:00+00:00
167932472,MAC004268,Std,2013-06-29 08:30:00,0.049,2013-06-29 08:30:00+00:00


In [11]:
n_hh = len(lcl_df['lcl_id'].unique())
print(f"Number of Households: {n_hh}")
print(f"Number of Standard Households: {len(lcl_df.loc[lcl_df["std_or_tou"]=="Std", "lcl_id"].unique())}")
print(f"Number of ToU Households: {len(lcl_df.loc[lcl_df["std_or_tou"]=="ToU", "lcl_id"].unique())}")

Number of Households: 5566
Number of Standard Households: 4443
Number of ToU Households: 1123


Make hours timezone aware and aggregate at the hourly level

In [ ]:
# make time to timezone aware
lcl_df['datetime'] = pd.to_datetime(lcl_df['datetime'], utc=False, errors='coerce')
lcl_df['datetime_utc'] = lcl_df['datetime'].dt.tz_localize(tz='UTC')

# make kwh per half hour numeric 
lcl_df['kwh_per_hh'] = pd.to_numeric(lcl_df['kwh_per_hh'], errors='coerce')  # coerce any 'Null' values to NaN

# group at the hour 
lcl_df['datetime_utc_hour'] = lcl_df['datetime_utc'].dt.floor('h')
lcl_df['n_obs'] = 1    # track how many time periods are included in the sum

# take *sum* of kWh per half hour
lcl_df_hourly = lcl_df.groupby(['datetime_utc_hour', 'lcl_id']).agg({
    'std_or_tou':'first', 
    'kwh_per_hh':'sum', 
    'n_obs':'sum'
}).reset_index()
lcl_df_hourly = lcl_df_hourly.rename(columns={'kwh_per_hh':'kwh', 
                                              'datetime_utc_hour':'datetime_utc'})

# if a half hour was missing, assume demand for the other half hour was the same as 
#lcl_df_hourly['kwh'] = lcl_df_hourly.apply(lambda row: row['kwh']*2 if row['n_obs']==1 else row['kwh'], axis=1)
mask = lcl_df_hourly['n_obs'] == 1
lcl_df_hourly.loc[mask, 'kwh'] *= 2


Restructure data to a size that can be handled in R

In [16]:
# exclude observations from 2014 that will not be used in the analysis
lcl_df_hourly = lcl_df_hourly[lcl_df_hourly['datetime_utc'].dt.year < 2014].copy()

In [17]:
# separate standard and dynamic time-of-use customers
lcl_df_hourly_std = lcl_df_hourly.loc[lcl_df_hourly['std_or_tou']=='Std'].copy()
lcl_df_hourly_tou = lcl_df_hourly.loc[lcl_df_hourly['std_or_tou']=='ToU'].copy()

# pivot wider so that each column corresponds to a separate household 
lcl_wide_std = lcl_df_hourly_std.pivot(columns='lcl_id', index='datetime_utc', values='kwh').reset_index()
lcl_wide_tou = lcl_df_hourly_tou.pivot(columns='lcl_id', index='datetime_utc', values='kwh').reset_index()

In [18]:
len(lcl_wide_std) == len(pd.date_range("2011-11-23 09:00", "2013-12-31 23:00", freq="h"))

True

## Export Dataset for Analysis in R

In [19]:
export_path = Path(os.path.join("data", "processed", "lcl_data"))
for name, df in {'Std':lcl_wide_std, 'ToU':lcl_wide_tou}.items():
    df.to_csv(os.path.join(export_path, f"lcl_hourly_demand_{name}.csv"))